In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 30.2 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

os.chdir('/content/drive/MyDrive/Colab Notebooks/Diego/Paper Interpretability')
print(os.getcwd())

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Diego/Paper Interpretability


In [4]:
from Model_utils.utils import get_segmented_data
from Model_utils.training_pipeline import run_optuna_experiment


In [ ]:
import warnings
import logging
import tensorflow as tf
import pickle
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from optuna.exceptions import ExperimentalWarning
    warnings.filterwarnings("ignore", category=ExperimentalWarning)
except Exception:
    pass

logging.getLogger("tensorflow").setLevel(logging.FATAL)
tf.get_logger().setLevel("ERROR")

# =========================================================
model_name = "eegnet"   # "eegnet", "shallowconvnet", "tgarnet"
db = "MI"               # "MI" o "TDAH"
tiem = "2.5-5"          # "0-7" o "2.5-5"

if tiem == "2.5-5":
    Samp = int(128 * 2.5)
else:
    Samp = int(128 * 7)

workers_by_model = {
    "eegnet": 11,
    "shallowconvnet": 11,
    "tgarnet": 12,
}

worker_id = 1

def split_subjects_evenly(subjects, total_workers, worker_id):
    """
    Reparte subjects en total_workers bloques lo más equilibrados posible
    y devuelve el bloque correspondiente a worker_id (1-indexado).
    """
    n = len(subjects)
    base = n // total_workers
    residuo = n % total_workers

    if worker_id < 1 or worker_id > total_workers:
        raise ValueError(
            f"worker_id debe estar entre 1 y {total_workers}, recibido: {worker_id}"
        )

    if worker_id <= residuo:
        inicio = (worker_id - 1) * (base + 1)
        fin = inicio + (base + 1)
    else:
        inicio = residuo * (base + 1) + (worker_id - residuo - 1) * base
        fin = inicio + base

    return subjects[inicio:fin]

with open(
    "Data/TDAH/folds.pkl",
    "rb"
) as f:
    folds = pickle.load(f)
    X, y, sbjs, _ = get_segmented_data(path_adhd='Data/TDAH/ieee/ADHD_group',
                                       path_control='Data/TDAH/ieee/Control_group')

if db == "MI":
    sujetos_validos = [
        1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,
        21, 22, 23, 24, 25, 26, 27, 28, 30, 31, 32, 33, 35, 36, 37, 38, 39, 40,
        41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52
    ]

    total_workers = workers_by_model[model_name]
    sujetos_asignados = split_subjects_evenly(
        subjects=sujetos_validos,
        total_workers=total_workers,
        worker_id=worker_id
    )

    print(f"Modelo: {model_name}")
    print(f"Worker {worker_id}/{total_workers}")
    print(f"Total sujetos asignados: {len(sujetos_asignados)}")
    print(f"Sujetos asignados: {sujetos_asignados}")

    for sj in sujetos_asignados:
        print(f"\nProcesando sujeto {sj} ...")

        study = run_optuna_experiment(
            model_name=model_name,
            data_mode="npz_folds",
            study_name=f"study_{model_name}_{db}_{tiem}_sj{sj}",
            journal_file=f"study_{model_name}_{db}_{tiem}_sj{sj}.journal",
            base_model_args={
                "nb_classes": 2,
                "Chans": 64,
                "Samples": Samp,
            },
            base_compile_args={},
            npz_path=f"Data/MI/{tiem}/subject_{sj}.npz",
            n_trials=20,
            epochs=100,
            batch_size=16,
            best_model_dir=f"{model_name}_{db}_{tiem}_best_models_{sj}"
        )

else:
    study = run_optuna_experiment(
        model_name=model_name,
        data_mode="subject_folds",
        study_name=f"study_{model_name}_{db}",
        journal_file=f"study_{model_name}_{db}.journal",
        base_model_args={
            "nb_classes": 2,
            "Chans": 19,
            "Samples": 512,
            "alpha": 2,
        },
        base_compile_args={},
        X=X,
        y=y,
        sbjs=sbjs,
        folds=folds,
        n_trials=20,
        epochs=100,
        batch_size=16,
        best_model_dir=f"{model_name}_best_models"
    )